# Task 3: Classification of ECG Beats Based on Patient Holdout Validation

In this notebook, we will:
1. Load the patient holdout training and testing data from CSV files
2. Train a Support Vector Machine (SVM) classifier on patients 100-234 (excluding test patients)
3. Test on completely unseen patients: 104, 113, 119, 208, 210
4. Evaluate and compare results with Beat Holdout method
5. Analyze the performance differences

## Key Difference from Task 2:
- **Beat Holdout**: Random split of beats from all patients → Data leakage possible
- **Patient Holdout**: Complete separation of patients → True generalization test

## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    confusion_matrix,
    classification_report
)
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set(style='whitegrid', palette='muted', font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 8)

## 2. Load Patient Holdout Data

### Held-out Patients for Testing:
- **Patient 104**
- **Patient 113**
- **Patient 119**
- **Patient 208**
- **Patient 210**

These 5 patients were completely excluded from training and will be used only for testing.

In [ ]:
# Load the patient holdout data
train_data = np.loadtxt('train_patients.csv', delimiter=',')
test_data = np.loadtxt('test_patients.csv', delimiter=',')

print(f"Training data shape: {train_data.shape}")
print(f"Testing data shape: {test_data.shape}")

# Define held-out patients
held_out_patients = [104, 113, 119, 208, 210]
print(f"\nHeld-out patients for testing: {held_out_patients}")

## 3. Prepare the Data

Extract features and labels, and verify patient separation.

In [ ]:
# Define class names
class_names = {
    0: 'Undetermined',
    1: 'Normal (N)',
    2: 'LBBBB (L)',
    3: 'RBBBB (R)',
    4: 'PVC (V)',
    5: 'APB (A)',
    6: 'Fusion VN (F)',
    7: 'Fusion PN (f)',
    8: 'Paced (/)'
}

# Extract features, labels, and patient numbers
X_train = train_data[:, :-2]  # ECG features (275 time points)
y_train = train_data[:, -2].astype(int)  # Class labels
patients_train = train_data[:, -1].astype(int)  # Patient numbers

X_test = test_data[:, :-2]
y_test = test_data[:, -2].astype(int)
patients_test = test_data[:, -1].astype(int)

print(f"\nFeature shape:")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"\nLabel shape:")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")

## 4. Verify Patient Separation

**Critical Check:** Ensure no overlap between training and testing patients.

In [ ]:
# Get unique patients in each set
unique_train_patients = np.unique(patients_train)
unique_test_patients = np.unique(patients_test)

print("Training patients:")
print(f"  Count: {len(unique_train_patients)}")
print(f"  IDs: {sorted(unique_train_patients)}")

print("\nTesting patients:")
print(f"  Count: {len(unique_test_patients)}")
print(f"  IDs: {sorted(unique_test_patients)}")

# Check for overlap (should be empty)
overlap = set(unique_train_patients).intersection(set(unique_test_patients))
print(f"\nPatient overlap: {overlap if overlap else 'None ✓'}")

if not overlap:
    print("✓ Patient separation verified: No data leakage!")
else:
    print("✗ WARNING: Patient overlap detected! Data leakage possible.")

## 5. Analyze Class Distribution

Check how classes are distributed in training and testing sets.

In [ ]:
print("\n" + "="*70)
print("CLASS DISTRIBUTION ANALYSIS")
print("="*70)

print("\nTraining set class distribution:")
unique_train, counts_train = np.unique(y_train, return_counts=True)
for cls, count in zip(unique_train, counts_train):
    percentage = (count / len(y_train)) * 100
    print(f"  Class {cls} ({class_names[cls]:20s}): {count:6d} samples ({percentage:5.2f}%)")

print(f"\nTotal training samples: {len(y_train)}")

print("\n" + "-"*70)

print("\nTesting set class distribution:")
unique_test, counts_test = np.unique(y_test, return_counts=True)
for cls, count in zip(unique_test, counts_test):
    percentage = (count / len(y_test)) * 100
    print(f"  Class {cls} ({class_names[cls]:20s}): {count:6d} samples ({percentage:5.2f}%)")

print(f"\nTotal testing samples: {len(y_test)}")

# Identify missing classes in test set
missing_classes = set(unique_train) - set(unique_test)
if missing_classes:
    print(f"\n⚠ Warning: Classes missing from test set: {missing_classes}")
    for cls in missing_classes:
        print(f"  - Class {cls} ({class_names[cls]})")

## 6. Visualize Class Distribution

In [ ]:
def plot_class_distribution(y_train, y_test, class_names_dict):
    """
    Plot class distribution for training and testing sets side by side.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Training distribution
    unique_train, counts_train = np.unique(y_train, return_counts=True)
    class_labels_train = [class_names_dict.get(i, f"Class {i}") for i in unique_train]
    
    axes[0].bar(range(len(unique_train)), counts_train, color='steelblue', alpha=0.8, edgecolor='black')
    axes[0].set_xlabel('Class', fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Number of Samples', fontsize=12, fontweight='bold')
    axes[0].set_title('Training Set Class Distribution\n(Patient Holdout)', 
                      fontsize=14, fontweight='bold', pad=15)
    axes[0].set_xticks(range(len(unique_train)))
    axes[0].set_xticklabels(class_labels_train, rotation=45, ha='right')
    axes[0].grid(axis='y', alpha=0.3, linestyle='--')
    
    # Add value labels on bars
    for i, count in enumerate(counts_train):
        axes[0].text(i, count + max(counts_train)*0.02, str(count), 
                    ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    # Testing distribution
    unique_test, counts_test = np.unique(y_test, return_counts=True)
    class_labels_test = [class_names_dict.get(i, f"Class {i}") for i in unique_test]
    
    axes[1].bar(range(len(unique_test)), counts_test, color='coral', alpha=0.8, edgecolor='black')
    axes[1].set_xlabel('Class', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Number of Samples', fontsize=12, fontweight='bold')
    axes[1].set_title('Testing Set Class Distribution\n(Patients: 104, 113, 119, 208, 210)', 
                      fontsize=14, fontweight='bold', pad=15)
    axes[1].set_xticks(range(len(unique_test)))
    axes[1].set_xticklabels(class_labels_test, rotation=45, ha='right')
    axes[1].grid(axis='y', alpha=0.3, linestyle='--')
    
    # Add value labels on bars
    for i, count in enumerate(counts_test):
        axes[1].text(i, count + max(counts_test)*0.02, str(count), 
                    ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

plot_class_distribution(y_train, y_test, class_names)

## 7. Feature Standardization

In [ ]:
# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature standardization complete.")
print(f"Training features - Mean: {X_train_scaled.mean():.6f}, Std: {X_train_scaled.std():.6f}")
print(f"Testing features - Mean: {X_test_scaled.mean():.6f}, Std: {X_test_scaled.std():.6f}")

## 8. Train SVM Classifier

Using the same SVM configuration as Task 2 for fair comparison.

In [ ]:
print("Training SVM classifier on patient holdout data...")
print("This may take several minutes...\n")

# Train SVM with same parameters as beat holdout for comparison
svm_classifier = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    random_state=42,
    verbose=True
)

svm_classifier.fit(X_train_scaled, y_train)

print("\nTraining complete!")

## 9. Make Predictions

In [ ]:
print("Making predictions on held-out patients...")
y_pred = svm_classifier.predict(X_test_scaled)
print("Predictions complete!")

## 10. Model Evaluation Function

Using the same comprehensive evaluation function as Task 2.

In [ ]:
def evaluate_model(y_true, y_pred, class_names_dict, model_name="Model"):
    """
    Comprehensive model evaluation function.
    
    Parameters:
    -----------
    y_true : array-like
        True labels
    y_pred : array-like
        Predicted labels
    class_names_dict : dict
        Dictionary mapping class IDs to class names
    model_name : str
        Name of the model for display
    
    Returns:
    --------
    metrics : dict
        Dictionary containing all computed metrics
    """
    
    # Get unique classes actually present in the data
    unique_classes = np.unique(np.concatenate([y_true, y_pred]))
    
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0, labels=unique_classes)
    precision_weighted = precision_score(y_true, y_pred, average='weighted', zero_division=0, labels=unique_classes)
    recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0, labels=unique_classes)
    recall_weighted = recall_score(y_true, y_pred, average='weighted', zero_division=0, labels=unique_classes)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0, labels=unique_classes)
    f1_weighted = f1_score(y_true, y_pred, average='weighted', zero_division=0, labels=unique_classes)
    
    # Per-class metrics
    precision_per_class = precision_score(y_true, y_pred, average=None, zero_division=0, labels=unique_classes)
    recall_per_class = recall_score(y_true, y_pred, average=None, zero_division=0, labels=unique_classes)
    f1_per_class = f1_score(y_true, y_pred, average=None, zero_division=0, labels=unique_classes)
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=unique_classes)
    
    # Print overall metrics
    print(f"\n{'='*60}")
    print(f"{model_name} - Evaluation Metrics")
    print(f"{'='*60}\n")
    
    print(f"Overall Metrics:")
    print(f"  Accuracy:           {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"\n  Macro Average:")
    print(f"    Precision:        {precision_macro:.4f}")
    print(f"    Recall:           {recall_macro:.4f}")
    print(f"    F1-Score:         {f1_macro:.4f}")
    print(f"\n  Weighted Average:")
    print(f"    Precision:        {precision_weighted:.4f}")
    print(f"    Recall:           {recall_weighted:.4f}")
    print(f"    F1-Score:         {f1_weighted:.4f}")
    
    # Print per-class metrics
    print(f"\n{'='*60}")
    print(f"Per-Class Metrics:")
    print(f"{'='*60}\n")
    
    print(f"{'Class':<20} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'Support'}")
    print(f"{'-'*70}")
    
    for i, cls in enumerate(unique_classes):
        class_name = class_names_dict.get(cls, f"Class {cls}")
        support = np.sum(y_true == cls)
        print(f"{class_name:<20} {precision_per_class[i]:<12.4f} {recall_per_class[i]:<12.4f} {f1_per_class[i]:<12.4f} {support}")
    
    # Store metrics in dictionary
    metrics = {
        'accuracy': accuracy,
        'precision_macro': precision_macro,
        'precision_weighted': precision_weighted,
        'recall_macro': recall_macro,
        'recall_weighted': recall_weighted,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'precision_per_class': precision_per_class,
        'recall_per_class': recall_per_class,
        'f1_per_class': f1_per_class,
        'confusion_matrix': cm,
        'unique_classes': unique_classes
    }
    
    return metrics

## 11. Evaluate the Model

In [ ]:
# Evaluate the SVM model
metrics = evaluate_model(y_test, y_pred, class_names, model_name="SVM (Patient Holdout)")

## 12. Visualize Confusion Matrix

In [ ]:
def plot_confusion_matrix(cm, class_names_dict, unique_classes, title='Confusion Matrix', figsize=(12, 10)):
    """
    Plot confusion matrix as a heatmap.
    """
    plt.figure(figsize=figsize)
    
    class_labels = [class_names_dict.get(i, f"Class {i}") for i in unique_classes]
    
    # Create heatmap
    sns.heatmap(
        cm, 
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_labels,
        yticklabels=class_labels,
        cbar_kws={'label': 'Count'},
        square=True,
        linewidths=0.5,
        linecolor='gray'
    )
    
    plt.title(title, fontsize=16, fontweight='bold', pad=20)
    plt.ylabel('True Label', fontsize=14, fontweight='bold')
    plt.xlabel('Predicted Label', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

# Plot confusion matrix
plot_confusion_matrix(
    metrics['confusion_matrix'], 
    class_names,
    metrics['unique_classes'],
    title='SVM Confusion Matrix (Patient Holdout Method)\nTest Patients: 104, 113, 119, 208, 210'
)

## 13. Visualize Per-Class Performance

In [ ]:
def plot_per_class_metrics(metrics, class_names_dict, unique_classes, figsize=(14, 6)):
    """
    Plot per-class precision, recall, and F1-score.
    """
    class_labels = [class_names_dict.get(i, f"Class {i}") for i in unique_classes]
    
    # Extract per-class metrics
    precision = metrics['precision_per_class']
    recall = metrics['recall_per_class']
    f1 = metrics['f1_per_class']
    
    # Create bar plot
    fig, ax = plt.subplots(figsize=figsize)
    
    x = np.arange(len(class_labels))
    width = 0.25
    
    bars1 = ax.bar(x - width, precision, width, label='Precision', color='#3498db', alpha=0.8)
    bars2 = ax.bar(x, recall, width, label='Recall', color='#2ecc71', alpha=0.8)
    bars3 = ax.bar(x + width, f1, width, label='F1-Score', color='#e74c3c', alpha=0.8)
    
    # Add value labels on bars
    def add_value_labels(bars):
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}',
                   ha='center', va='bottom', fontsize=9)
    
    add_value_labels(bars1)
    add_value_labels(bars2)
    add_value_labels(bars3)
    
    ax.set_xlabel('Class', fontsize=12, fontweight='bold')
    ax.set_ylabel('Score', fontsize=12, fontweight='bold')
    ax.set_title('Per-Class Performance Metrics (SVM - Patient Holdout)', 
                fontsize=14, fontweight='bold', pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(class_labels, rotation=45, ha='right')
    ax.legend(loc='lower right', fontsize=10)
    ax.set_ylim([0, 1.1])
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.show()

# Plot per-class metrics
plot_per_class_metrics(metrics, class_names, metrics['unique_classes'])

## 14. Per-Patient Performance Analysis

Analyze performance separately for each held-out patient.

In [ ]:
print("\n" + "="*70)
print("PER-PATIENT PERFORMANCE ANALYSIS")
print("="*70)

for patient_id in held_out_patients:
    # Get indices for this patient
    patient_mask = patients_test == patient_id
    
    if not np.any(patient_mask):
        print(f"\nPatient {patient_id}: No data")
        continue
    
    # Get predictions for this patient
    y_true_patient = y_test[patient_mask]
    y_pred_patient = y_pred[patient_mask]
    
    # Calculate accuracy
    accuracy_patient = accuracy_score(y_true_patient, y_pred_patient)
    
    # Get class distribution
    unique_classes_patient, counts = np.unique(y_true_patient, return_counts=True)
    
    print(f"\nPatient {patient_id}:")
    print(f"  Total beats: {len(y_true_patient)}")
    print(f"  Accuracy: {accuracy_patient:.4f} ({accuracy_patient*100:.2f}%)")
    print(f"  Classes present:")
    for cls, count in zip(unique_classes_patient, counts):
        print(f"    - Class {cls} ({class_names[cls]}): {count} beats")

## 15. Classification Report

In [ ]:
# Get unique classes from test set
unique_classes = metrics['unique_classes']
target_names = [class_names[i] for i in unique_classes]

print("\nDetailed Classification Report:")
print("="*80)
print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0, labels=unique_classes))

## 16. Comparison: Beat Holdout vs Patient Holdout

### Expected Differences:

| Aspect | Beat Holdout | Patient Holdout |
|--------|--------------|------------------|
| **Accuracy** | Higher (95-99%) | Lower (75-90%) |
| **Generalization** | Overestimated | True generalization |
| **Data Leakage** | Yes (same patients in train/test) | No (complete separation) |
| **Clinical Relevance** | Limited | High (real-world scenario) |
| **Class Balance** | Good (resampled) | Poor (some classes missing) |
| **Use Case** | Algorithm comparison | Deployment readiness |

### Why Performance Drops:

1. **No Patient-Specific Learning**: Model cannot rely on patient-specific morphology patterns
2. **True Inter-Patient Variability**: Must generalize across different:
   - Electrode placements
   - Body compositions
   - Medications
   - Pre-existing conditions
3. **Missing Classes**: Some arrhythmia types may be unique to held-out patients
4. **Smaller Test Set**: Only 5 patients for testing vs all patients in beat holdout

### Clinical Interpretation:

**Patient Holdout Performance** better represents:
- How the model will perform on **new patients** in a hospital
- Whether the model learned **general ECG patterns** vs **patient-specific quirks**
- The model's **true clinical utility**

## 17. Key Observations and Discussion

### Summary:
- **Method**: Patient Holdout Validation
- **Training**: 42 patients
- **Testing**: 5 held-out patients (104, 113, 119, 208, 210)
- **Classifier**: SVM with RBF kernel

### Expected Findings:

1. **Lower Overall Accuracy**: Performance will drop compared to beat holdout
   - This is **expected and desired** - it shows true generalization

2. **Class Imbalance Issues**: Some classes may be missing or underrepresented
   - Certain arrhythmias are rare and patient-specific
   - Classes L, R, A might be missing from test set

3. **High Variability**: Performance may vary significantly between patients
   - Some patients easier to classify than others
   - Patient-specific factors affect difficulty

4. **Normal Class Dominance**: Normal beats (N) typically classified well
   - Most common and consistent across patients

### Clinical Implications:

- **Patient holdout is the gold standard** for medical AI validation
- Lower accuracy here doesn't mean the model is "worse" - it means we have **realistic expectations**
- This method reveals whether the model can truly help with **new, unseen patients**

### Next Steps:
- **Task 4**: Apply explainability techniques to understand what the model learned
- **Task 5**: Try different classifiers and compare their patient holdout performance
- **Task 6**: Use clustering to understand patient-specific patterns

## 18. Save Model and Results

In [ ]:
import pickle

# Save model
with open('svm_patient_holdout_model.pkl', 'wb') as f:
    pickle.dump(svm_classifier, f)
    
# Save scaler
with open('scaler_patient_holdout.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save metrics
with open('metrics_patient_holdout.pkl', 'wb') as f:
    pickle.dump(metrics, f)

print("Model, scaler, and metrics saved successfully!")
print("\nSaved files:")
print("  - svm_patient_holdout_model.pkl")
print("  - scaler_patient_holdout.pkl")
print("  - metrics_patient_holdout.pkl")